In [0]:
VOLUME_PATH = "/Volumes/olist_project/olist/raw_data"

files = {

    "orders":        "olist_orders_dataset.csv",

    "customers":     "olist_customers_dataset.csv",

    "order_items":   "olist_order_items_dataset.csv",

    "order_payments":"olist_order_payments_dataset.csv",

    "order_reviews": "olist_order_reviews_dataset.csv",

    "products":      "olist_products_dataset.csv",

    "sellers":       "olist_sellers_dataset.csv",

    "geolocation":   "olist_geolocation_dataset.csv",

    "category_translation": "product_category_name_translation.csv"

}
 

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS olist_project.bronze
COMMENT 'Raw ingestion layer - no transformations applied';
 

In [0]:
for table_name, file_name in files.items():
    file_path = f"{VOLUME_PATH}/{file_name}"
    df = spark.read \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .option("multiLine", "true") \
        .option("escape", '"') \
        .csv(file_path)
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"olist_project.bronze.{table_name}")
    print(f"Loaded: {table_name} — {df.count()} rows")
 

Loaded: orders — 99441 rows
Loaded: customers — 99441 rows
Loaded: order_items — 112650 rows
Loaded: order_payments — 103886 rows
Loaded: order_reviews — 99224 rows
Loaded: products — 32951 rows
Loaded: sellers — 3095 rows
Loaded: geolocation — 1000163 rows
Loaded: category_translation — 71 rows


In [0]:
%sql
SHOW TABLES IN olist_project.bronze;

database,tableName,isTemporary
bronze,category_translation,false
bronze,customers,false
bronze,geolocation,false
bronze,order_items,false
bronze,order_payments,false
bronze,order_reviews,false
bronze,orders,false
bronze,products,false
bronze,sellers,false


In [0]:
for table_name in files.keys():
    print(f"\n--- {table_name} ---")
    spark.sql(f"SELECT * FROM olist_project.bronze.{table_name} LIMIT 3").display(truncate=False)


--- orders ---


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z



--- customers ---


customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP



--- order_items ---


...

seller_id,seller_zip_code_prefix,seller_city,seller_state
3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ



--- geolocation ---


geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1037,-23.54562128115268,-46.63929204800168,sao paulo,SP
1046,-23.546081127035535,-46.64482029837157,sao paulo,SP
1046,-23.54612896641469,-46.64295148361138,sao paulo,SP



--- category_translation ---


product_category_name,product_category_name_english
beleza_saude,health_beauty
informatica_acessorios,computers_accessories
automotivo,auto


In [0]:
%sql
SELECT 'orders'        AS table_name, COUNT(*) AS row_count FROM olist_project.bronze.orders        UNION ALL
SELECT 'customers',                   COUNT(*)              FROM olist_project.bronze.customers       UNION ALL
SELECT 'order_items',                 COUNT(*)              FROM olist_project.bronze.order_items     UNION ALL
SELECT 'order_payments',              COUNT(*)              FROM olist_project.bronze.order_payments  UNION ALL
SELECT 'order_reviews',               COUNT(*)              FROM olist_project.bronze.order_reviews   UNION ALL
SELECT 'products',                    COUNT(*)              FROM olist_project.bronze.products        UNION ALL
SELECT 'sellers',                     COUNT(*)              FROM olist_project.bronze.sellers         UNION ALL
SELECT 'geolocation',                 COUNT(*)              FROM olist_project.bronze.geolocation     UNION ALL
SELECT 'category_translation',        COUNT(*)              FROM olist_project.bronze.category_translation
ORDER BY row_count DESC;

table_name,row_count
geolocation,1000163
order_items,112650
order_payments,103886
orders,99441
customers,99441
order_reviews,99224
products,32951
sellers,3095
category_translation,71


In [0]:
%sql
DESCRIBE HISTORY olist_project.bronze.orders;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-03-25T07:18:05.000Z,71960221862429,akshaysuhas.chavan@tcs.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(351850165390627),c1015161-a27c-4327-be09-1f3239945e0f,0325-071512-5l1vkep6-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 99441, numOutputBytes -> 6392817)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
